In [5]:
!pip install feedparser newspaper3k transformers torch lxml_html_clean --quiet

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.3 MB/s eta 0:00:00


In [6]:
import feedparser
from newspaper import Article
from transformers import pipeline
from datetime import datetime
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
from transformers import pipeline

summarizer = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Model Loaded Successfully


In [33]:
rss_feeds = [
    "https://feeds.bbci.co.uk/news/rss.xml",
    "https://rss.cnn.com/rss/edition.rss",
    "https://feeds.reuters.com/reuters/topNews",
    "https://www.thehindu.com/news/national/feeder/default.rss",
    "https://timesofindia.indiatimes.com/rssfeeds/-2128936835.cms"
]

print("RSS feeds loaded")

RSS feeds loaded


In [9]:
article_links = []

for feed_url in rss_feeds:
    feed = feedparser.parse(feed_url)

    for entry in feed.entries[:10]:
        if entry.link not in article_links:
            article_links.append(entry.link)

print(f"Total Articles Collected: {len(article_links)}")

Total Articles Collected: 30


In [32]:
news_articles = []

keywords_to_ignore = [
    "advertisement",
    "sponsored",
    "watch live",
    "video",
    "photos"
]

for link in article_links:
    try:
        article = Article(link)

        article.download()
        article.parse()

        title = article.title.strip()
        text = article.text.strip()

        # Ignore weak/noisy articles
        if (
            len(text) < 500
            or any(word.lower() in title.lower() for word in keywords_to_ignore)
        ):
            continue

        news_articles.append({
            "title": title,
            "text": text[:2000],
            "link": link
        })

        print("Collected:", title)

    except:
        pass

print(f"\nTotal Clean Articles: {len(news_articles)}")

Collected: Jonathan Gjoshe: Footballer in mass train attack reveals he was stabbed seven times
Collected: Am I part of the luckiest generation alive?
Collected: US PGA Championship 2026: England's Aaron Rai wins first major at Aronimink
Collected: Karnataka man kills minor daughter over marriage dispute by pushing her into well
Collected: Virudhunagar AIADMK unit stands by Edappadi Palaniswami: Rajenthra Bhalaji
Collected: Veteran AIADMK leader Semmalai quits party
Collected: TMC MLA's son held with firearms in Bengal
Collected: Census 2027: Over 11 lakh households in Maharashtra complete self-enumeration
Collected: Return of Leiden copper plates should spark efforts for further repatriations, say Indian archaeologists
Collected: Chola-era Anaimangalam Plates, in possession of Leiden University since 1862, returned to India
Collected: DMK urges Centre to hand over Anaimangalam copper plates to Tamil Nadu
Collected: Demand to bring Anaimangalam Chola-era copper plates to Tamil Nadu gath

In [34]:
combined_news = ""

for article in news_articles:
    combined_news += article["title"] + "\n"
    combined_news += article["text"][:1500] + "\n\n"

print("Combined News Length:", len(combined_news))

Combined News Length: 19050


In [35]:
from transformers import pipeline

# -----------------------------------
# LOAD AI CLASSIFIER
# -----------------------------------

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# -----------------------------------
# CATEGORY LABELS
# -----------------------------------

candidate_labels = [
    "Global News",
    "Business & Economy",
    "Technology & AI",
    "India"
]

# -----------------------------------
# EMPTY CATEGORY SECTIONS
# -----------------------------------

categorized_news = {
    "🌍 GLOBAL NEWS": [],
    "💼 BUSINESS & ECONOMY": [],
    "🤖 TECHNOLOGY & AI": [],
    "🇮🇳 INDIA": [],
    "📌 OTHER IMPORTANT UPDATES": []
}

# -----------------------------------
# FORCE GLOBAL NEWS KEYWORDS
# -----------------------------------

global_keywords = [
    "war", "ukraine", "russia", "china",
    "israel", "gaza", "nato", "iran",
    "world", "global", "europe", "un",
    "united nations", "military", "attack",
    "conflict", "ceasefire", "tariff"
]

# -----------------------------------
# CLASSIFY ARTICLES
# -----------------------------------

for article in news_articles:

    headline = article["title"]
    headline_lower = headline.lower()

    assigned = False

    # -----------------------------------
    # FORCE GLOBAL NEWS
    # -----------------------------------

    if any(keyword in headline_lower for keyword in global_keywords):

        categorized_news["🌍 GLOBAL NEWS"].append(article)
        assigned = True

    # -----------------------------------
    # AI CLASSIFICATION
    # -----------------------------------

    if not assigned:

        try:

            result = classifier(
                headline,
                candidate_labels
            )

            category = result["labels"][0]

        except Exception:
            category = "Other"

        # -----------------------------------
        # MAP AI CATEGORY
        # -----------------------------------

        if category == "Business & Economy":

            categorized_news["💼 BUSINESS & ECONOMY"].append(article)

        elif category == "Technology & AI":

            categorized_news["🤖 TECHNOLOGY & AI"].append(article)

        elif category == "India":

            categorized_news["🇮🇳 INDIA"].append(article)

        elif category == "Global News":

            categorized_news["🌍 GLOBAL NEWS"].append(article)

        else:

            categorized_news["📌 OTHER IMPORTANT UPDATES"].append(article)

# -----------------------------------
# GENERATE FINAL NEWS BRIEFING
# -----------------------------------

daily_summary = ""

section_order = [
    "🌍 GLOBAL NEWS",
    "💼 BUSINESS & ECONOMY",
    "🤖 TECHNOLOGY & AI",
    "🇮🇳 INDIA",
    "📌 OTHER IMPORTANT UPDATES"
]

for category in section_order:

    articles = categorized_news[category]

    if len(articles) == 0:
        continue

    daily_summary += f"\n{category}\n\n"

    for article in articles[:5]:

        daily_summary += (
            f"• {article['title']}\n"
            f"  {article['link']}\n\n"
        )

print("\n📰 DAILY AI NEWS BRIEFING\n")
print(daily_summary)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


📰 DAILY AI NEWS BRIEFING


🌍 GLOBAL NEWS

• Jonathan Gjoshe: Footballer in mass train attack reveals he was stabbed seven times
  https://www.bbc.com/sport/football/articles/cp8py1n500lo?at_medium=RSS&at_campaign=rss

• Virudhunagar AIADMK unit stands by Edappadi Palaniswami: Rajenthra Bhalaji
  https://www.thehindu.com/news/national/tamil-nadu/virudhunagar-aiadmk-unit-stands-by-eps-says-rajendhra-bhalaji/article70990974.ece

• Chola-era Anaimangalam Plates, in possession of Leiden University since 1862, returned to India
  https://www.thehindu.com/news/national/chola-era-anaimangalam-plates-in-possession-of-leiden-university-since-1862-returned-to-india/article70987678.ece


💼 BUSINESS & ECONOMY

• Am I part of the luckiest generation alive?
  https://www.bbc.com/news/articles/cj6pyk7e3w4o?at_medium=RSS&at_campaign=rss

• Jobs, AI, trade & more: Key outcomes of PM Modi’s Sweden visit
  https://timesofindia.indiatimes.com/india/jobs-ai-trade-more-key-outcomes-of-pm-modis-sweden-visit/

In [36]:
today = datetime.now().strftime("%Y-%m-%d")

filename = f"daily_news_summary_{today}.txt"

with open(filename, "w", encoding="utf-8") as f:
    f.write(daily_summary)

print(f"Summary saved as: {filename}")

Summary saved as: daily_news_summary_2026-05-18.txt


In [38]:
from google.colab import files

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Your Gmail
sender_email = "ruhikakriti08@gmail.com"
receiver_email = "ruhikakriti08@gmail.com"

# Your 16-character Gmail App Password
password = "wmoqjuqyvhzoaxci"

# Create email message
message = MIMEMultipart("alternative")

message["Subject"] = "📰 Daily AI News Briefing"
message["From"] = sender_email
message["To"] = receiver_email

# HTML Email Design
html = f"""
<html>

<body style="font-family: Arial; padding: 20px; background-color: #f4f4f4;">

<div style="
    background-color: white;
    padding: 25px;
    border-radius: 10px;
    max-width: 800px;
    margin: auto;
    box-shadow: 0px 0px 10px rgba(0,0,0,0.1);
">

<h1 style="color: #222;">
📰 Daily AI News Briefing
</h1>

<p style="color: #666;">
Your automated AI-generated news summary for today.
</p>

<hr>

<pre style="
    font-size: 15px;
    white-space: pre-wrap;
    line-height: 1.6;
    color: #333;
">
{daily_summary}
</pre>

<hr>

<p style="font-size: 13px; color: gray;">
Generated automatically using your AI News Agent
</p>

</div>

</body>
</html>
"""

# Attach HTML email
message.attach(MIMEText(html, "html"))

# Send Email
try:
    server = smtplib.SMTP("smtp.gmail.com", 587)

    server.ehlo()
    server.starttls()
    server.ehlo()

    server.login(sender_email, password)

    server.sendmail(
        sender_email,
        receiver_email,
        message.as_string()
    )

    server.quit()

    print("✅ Email sent successfully!")

except Exception as e:
    print("❌ Error:", e)

✅ Email sent successfully!
